# Análisis interno de ESM-2: Proyecto Completo

Trabajo de Segundo Corte - Ciencia de Datos  
Universidad de Pamplona - Ingeniería de Sistemas

**Autor:** Anderson González (@Albonire)  
**Modelo:** facebook/esm2_t6_8M_UR50D

Este notebook integra todas las actividades de análisis:
1. Marco teórico.
2. Ejecución y extracción de formas.
3. Inspección del código fuente.
4. Visualización de embeddings y similitud.
5. Análisis de matrices de atención.
6. Masked Language Modeling (MLM).
7. Usos reales y limitaciones.


## 1. Marco teórico

El Transformer original (Vaswani et al., 2017) revolucionó el procesamiento de secuencias mediante el mecanismo de self-attention. ESM-2 es un modelo encoder-only que procesa la secuencia completa de forma bidireccional.


In [ ]:
%pip install -q torch transformers matplotlib scikit-learn pandas tabulate


In [ ]:
import os
import inspect
import textwrap
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM
from transformers.models.esm import modeling_esm

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
SEQUENCES = {
    "Original": "MKTAYIAKQRQISFVKSHFSRQDILD",
    "Mutada": "MKTAFIAKQRQISFVKSHFSRQDILD",
    "Alterada": "DLIDQRSFHSSKVFSIQRQKAIYATKM",
}


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
try:
    model = EsmModel.from_pretrained(MODEL_NAME, attn_implementation="eager")
except:
    model = EsmModel.from_pretrained(MODEL_NAME)
model.eval()

mlm_model = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()


## 2. Ejecución y formas internas


In [ ]:
def analyze(name, seq):
    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, output_attentions=True)
    
    print(f"Secuencia: {name}")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
    print(f"Shape last_hidden_state: {out.last_hidden_state.shape}")
    return inputs, out

results = {n: analyze(n, s) for n, s in SEQUENCES.items()}


## 3. Inspección del código fuente


In [ ]:
print(f"Clase del modelo: {type(model)}")
print(f"Embeddings: {type(model.embeddings)}")
print(f"Capas: {len(model.encoder.layer)}")


## 4. Embeddings y Similitud


In [ ]:
def get_global(out):
    return out.last_hidden_state[0, 1:-1, :].mean(dim=0).numpy()

glob_embs = {n: get_global(r[1]) for n, r in results.items()}
names = list(glob_embs.keys())
matrix = np.array([glob_embs[n] for n in names])
sim = cosine_similarity(matrix)
pd.DataFrame(sim, index=names, columns=names)


## 5. Matrices de Atención


In [ ]:
def plot_att(att, tokens, title):
    plt.figure(figsize=(8,7))
    plt.imshow(att, aspect="auto")
    plt.title(title)
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.colorbar()
    plt.show()

if results["Original"][1].attentions:
    att = results["Original"][1].attentions[0][0, 0].detach().numpy()
    tokens = tokenizer.convert_ids_to_tokens(results["Original"][0]["input_ids"][0])
    plot_att(att, tokens, "Atención Capa 0, Cabeza 0")


## 6. Masked Language Modeling


In [ ]:
masked_seq = SEQUENCES["Original"].replace("Y", tokenizer.mask_token, 1)
inputs_m = tokenizer(masked_seq, return_tensors="pt")
mask_idx = (inputs_m.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

with torch.no_grad():
    logits = mlm_model(**inputs_m).logits
    
top5 = torch.topk(logits[0, mask_idx, :], 5)
print(f"Posición enmascarada predicha:")
for s, i in zip(top5.values[0], top5.indices[0]):
    print(f"Token: {tokenizer.convert_ids_to_tokens([i.item()])[0]} | Score: {s:.4f}")


## 7. Usos reales y limitaciones

ESM-2 tiene usos en comparación semántica, análisis de mutaciones y apoyo a la predicción estructural. Sus limitaciones incluyen la incapacidad de probar causalidad biológica y la necesidad de validación científica experimental.
